# Día 5 — PCA, selección e ingeniería de variables

Reducir dimensiones sin perder de vista qué información y significado conservamos.

## Objetivos de aprendizaje

- Explicar escala, componentes, loadings y varianza explicada.
- Visualizar observaciones en dos dimensiones con cautela.
- Comparar predicción con y sin PCA evitando leakage.
- Distinguir selección, transformación e ingeniería de variables.

> **Tiempo del plan oficial:** 3 horas sincrónicas. Prioriza PCA, lectura de componentes, selección de variables e ingeniería; la optimización sistemática continúa en el Día 6.

**Dataset:** indicadores país-año del Banco Mundial, archivo descargado en `datos/world_bank_latam_ml.csv`; sin llamadas a APIs durante el notebook.

## Cómo estudiar esta jornada

Lee primero la pregunta y la explicación; ejecuta luego las celdas en orden. Después de cada tabla o gráfico, detente en **Interpreta** y redacta una conclusión con evidencia. Los resultados dependen de la versión de las librerías y de las particiones; usa tus propios números y no copies los ejemplos como verdades universales. No confundas predicción con causalidad.

## 1. Pregunta: muchas variables, menos dimensiones

Usaremos indicadores macroeconómicos país-año. PCA crea combinaciones lineales que resumen variación; no selecciona las variables «más importantes» para un target. Separaremos dos usos: **visualizar estructura** y **predecir crecimiento**. Siempre explicaremos cuánta información se pierde.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

macro = pd.read_csv(DATA_DIR / "world_bank_latam_ml.csv")
features = ["inflacion_pct", "desempleo_pct", "exportaciones_pct_pib",
            "inversion_pct_pib", "crecimiento_poblacion_pct"]
target = "crecimiento_pib_pct"
df = macro.dropna(subset=features + [target]).copy()
print("Observaciones completas:", len(df), "Países:", df.pais.nunique())
df[features + [target]].describe().round(2).T


### Revisa escalas y dependencias

Las variables mezclan tasas y proporciones con magnitudes diferentes. PCA maximiza varianza, así que sin estandarizar una variable de gran dispersión podría dominar el primer componente por unidades. También hay posible dependencia entre observaciones del mismo país a lo largo de años; una división aleatoria sirve como demostración, pero no estima con rigor el desempeño en un país nuevo ni en el futuro.

In [ ]:
print("Correlaciones entre predictores:")
print(df[features].corr().round(2))
print("Desviaciones originales:")
print(df[features].std().round(2))


## 2. PCA para visualización y varianza explicada

Estandarizamos y ajustamos PCA para explorar la geometría completa. En una tarea predictiva, scaler y PCA deben aprenderse solo en Train. Para una visualización descriptiva de toda la muestra no hay evaluación predictiva, pero no uses este gráfico para afirmar desempeño fuera de muestra.

In [ ]:
scaler_viz = StandardScaler()
Xz = scaler_viz.fit_transform(df[features])
pca = PCA().fit(Xz)
var = pd.DataFrame({"componente": np.arange(1, len(features)+1),
                    "varianza": pca.explained_variance_ratio_,
                    "acumulada": np.cumsum(pca.explained_variance_ratio_)})
var.round(3)


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(var.componente, var.varianza, alpha=.65, label="Individual")
plt.plot(var.componente, var.acumulada, marker="o", color="darkred", label="Acumulada")
plt.axhline(.8, color="gray", linestyle="--", label="80 % (referencia, no regla)")
plt.xlabel("Componente principal"); plt.ylabel("Proporción de varianza")
plt.title("PCA: varianza explicada"); plt.xticks(var.componente); plt.legend(); plt.grid(axis="y", alpha=.2)


### Interpreta varianza y loadings

La varianza acumulada responde qué fracción de la variabilidad de **estas X estandarizadas** resume el conjunto de componentes. Un umbral como 80 % es orientativo, no una garantía de que se conserve la información predictiva. Inspecciona loadings: signos dependen de la orientación arbitraria del eje; importa qué variables contribuyen y cómo se relacionan.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=features,
                        columns=[f"PC{i+1}" for i in range(len(features))])
print("Loadings PC1-PC2:")
print(loadings.iloc[:, :2].round(3))
Z2 = pca.transform(Xz)[:, :2]
plt.figure(figsize=(8, 5))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=df[target], cmap="coolwarm", alpha=.75)
plt.colorbar(sc, label="Crecimiento del PIB (%)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("País-año en los dos primeros componentes")


### Lee la proyección con cuidado

Cada punto es una observación país-año; el color muestra el crecimiento observado solo para ayudar a explorar. PC1/PC2 son combinaciones, no «inflación» y «desempleo». El gráfico puede revelar agrupación, gradiente o años extremos; no prueba que PCA prediga bien crecimiento ni que una variable cause otra. Los puntos del mismo país no son independientes.

## 3. PCA dentro de una evaluación predictiva

Para comparar predicción con/sin reducción, reservamos un test aleatorio estratificado por país-año solo como ejercicio. En trabajo temporal separaríamos años cronológicamente; para generalizar a otros países usaríamos grupos por país. PCA y escalado quedan dentro de cada Pipeline para evitar leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=.25, random_state=42)
sin_pca = make_pipeline(StandardScaler(), LinearRegression())
con_pca = make_pipeline(StandardScaler(), PCA(n_components=2), LinearRegression())
sin_pca.fit(X_train, y_train); con_pca.fit(X_train, y_train)
comparacion = []
for nombre, modelo in [("Lineal, 5 variables", sin_pca), ("PCA, 2 componentes", con_pca)]:
    pred = modelo.predict(X_test)
    comparacion.append({"modelo": nombre, "MAE": mean_absolute_error(y_test, pred), "R2": r2_score(y_test, pred)})
pd.DataFrame(comparacion).set_index("modelo").round(3)


### Resultado predictivo: no hay ganador automático

MAE se interpreta en puntos porcentuales de crecimiento; menor es mejor. $R^2$ puede ser negativo cuando el modelo se equivoca más que una referencia basada en la media del test. Compara las métricas de tu ejecución: comprimir a dos componentes puede reducir ruido o eliminar señal útil. Este test aleatorio puede ser optimista por la repetición de países; no es una evaluación de forecasting ni de generalización a un país nuevo.

## 4. Selección de variables: una pregunta distinta

PCA cambia la representación. La selección conserva variables originales. Combinaremos criterio sustantivo con importancia por permutación: cuánto empeora una métrica al barajar una columna. Importancia predictiva no es causalidad y puede repartirse entre predictores correlacionados.

In [ ]:
# Permutation importance en el modelo sin PCA y el conjunto de prueba.
imp = permutation_importance(sin_pca, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=20, random_state=42)
tabla_imp = pd.DataFrame({"variable": features, "aumento_MAE": imp.importances_mean,
                          "desviacion": imp.importances_std}).sort_values("aumento_MAE", ascending=False)
tabla_imp.round(3)


### Interpreta selección

Aumento positivo de MAE sugiere que el modelo depende predictivamente de esa variable en este conjunto; valores cercanos a cero o negativos pueden indicar poca contribución, redundancia o inestabilidad. No borres automáticamente columnas por un ranking de una sola partición. Contrasta con significado económico, correlación y estabilidad.

## 5. Ingeniería de características

Podemos transformar y combinar predictores si existe una hipótesis: por ejemplo, permitir curvatura entre inflación y desempleo mediante un término cuadrático. Más columnas aumentan flexibilidad y riesgo de sobreajuste. El preprocesamiento debe permanecer en Pipeline y la decisión se compara fuera de muestra.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df[features], df[target], test_size=.25, random_state=42)
lineal = make_pipeline(StandardScaler(), LinearRegression())
polinomico = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Ridge(alpha=10))
lineal.fit(X_tr, y_tr); polinomico.fit(X_tr, y_tr)
pd.DataFrame([{"modelo":"Lineal", "MAE":mean_absolute_error(y_te,lineal.predict(X_te)), "R2":r2_score(y_te,lineal.predict(X_te))},
              {"modelo":"Grado 2 + Ridge", "MAE":mean_absolute_error(y_te,polinomico.predict(X_te)), "R2":r2_score(y_te,polinomico.predict(X_te))}]).set_index("modelo").round(3)


### Cierre: decisión basada en evidencia

Resume: (a) cuánta varianza retienen dos componentes; (b) qué representan PC1/PC2 según loadings; (c) cómo cambia el error predictivo con PCA; (d) qué variable muestra mayor importancia por permutación; (e) por qué la validación aleatoria no basta para un uso de política o pronóstico. Evita afirmar que un componente «explica» causalmente crecimiento.

## Actividad autónoma — PCA y selección (20 puntos sugeridos)

Entrega un notebook con: gráfico PC1/PC2 explicado; tabla de varianza individual/acumulada; loadings interpretados; comparación de un modelo original y uno reducido con el mismo split; importancia por permutación; y un párrafo de límites por país, tiempo, datos faltantes y asociación/causalidad. Incluye la fuente local y no uses datos institucionales.